# Colab + Google Drive Learning Cycle (GPU Template)

This template is tuned for GPU runtime on Colab.

Workflow:

1. Mount Google Drive
2. Prepare repository and dependencies
3. Verify GPU availability (`nvidia-smi`, `torch.cuda`)
4. Run learning/prediction with Drive data
5. Keep artifacts in Drive


## 0. Runtime setting

Before running, set Colab runtime to GPU:

- Runtime -> Change runtime type -> Hardware accelerator: **GPU**


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os

REPO_DIR = '/content/kyotei_Prediction'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/takajin0114/kyotei_Prediction.git {REPO_DIR}

%cd {REPO_DIR}
!python -m pip install --upgrade pip
!pip install -r requirements.txt


In [ ]:
# GPU check
!nvidia-smi

import torch
print('torch version:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('cuda device:', torch.cuda.get_device_name(0))


In [ ]:
import os

# Optional CUDA memory behavior tuning for long runs
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

DRIVE_ROOT = '/content/drive/MyDrive/kyotei_prediction'
DATA_DIR = f'{DRIVE_ROOT}/data/raw'
YEAR_MONTH = '2026-02'
PREDICT_DATE = '2026-02-14'

# Presets:
# - quick_check: shortest validation run
# - night_train: longer overnight training run
PROFILES = {
    'quick_check': {
        'N_TRIALS': 1,
        'MINIMAL': True,
        'RUN_FETCH': False,
        'FETCH_START_DATE': '2026-02-01',
        'FETCH_END_DATE': '2026-02-14',
        'RUN_PREDICTION': True,
    },
    'night_train': {
        'N_TRIALS': 20,
        'MINIMAL': False,
        'RUN_FETCH': False,
        'FETCH_START_DATE': '2026-01-01',
        'FETCH_END_DATE': '2026-02-14',
        'RUN_PREDICTION': True,
    },
}

PROFILE_NAME = 'quick_check'  # <- change to 'night_train' for longer run
if PROFILE_NAME not in PROFILES:
    raise ValueError(f'unknown PROFILE_NAME: {PROFILE_NAME}')

profile = PROFILES[PROFILE_NAME]
N_TRIALS = profile['N_TRIALS']
MINIMAL = profile['MINIMAL']
RUN_FETCH = profile['RUN_FETCH']
FETCH_START_DATE = profile['FETCH_START_DATE']
FETCH_END_DATE = profile['FETCH_END_DATE']
RUN_PREDICTION = profile['RUN_PREDICTION']

# Optional manual overrides
# N_TRIALS = 5
# MINIMAL = False

os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

print('PROFILE_NAME    =', PROFILE_NAME)
print('DRIVE_ROOT      =', DRIVE_ROOT)
print('DATA_DIR        =', DATA_DIR)
print('YEAR_MONTH      =', YEAR_MONTH)
print('PREDICT_DATE    =', PREDICT_DATE)
print('N_TRIALS        =', N_TRIALS)
print('MINIMAL         =', MINIMAL)
print('RUN_FETCH       =', RUN_FETCH)
print('FETCH_START_DATE=', FETCH_START_DATE)
print('FETCH_END_DATE  =', FETCH_END_DATE)
print('RUN_PREDICTION  =', RUN_PREDICTION)


In [ ]:
# Optional: fetch/update raw data directly into Drive.
# RUN_FETCH / date range are controlled by the selected profile.

if RUN_FETCH:
    !python -m kyotei_predictor.tools.batch.batch_fetch_all_venues \
      --start-date "{FETCH_START_DATE}" \
      --end-date "{FETCH_END_DATE}" \
      --stadiums ALL \
      --output-data-dir "{DATA_DIR}"


In [ ]:
import subprocess

cmd = [
    'python',
    'scripts/run_colab_learning_cycle.py',
    '--drive-root', DRIVE_ROOT,
    '--data-dir', DATA_DIR,
    '--year-month', YEAR_MONTH,
    '--n-trials', str(N_TRIALS),
    '--predict-date', PREDICT_DATE,
]
if MINIMAL:
    cmd.append('--minimal')
if not RUN_PREDICTION:
    cmd.append('--skip-prediction')

print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
import json
import os

pred_path = f'outputs/predictions_{PREDICT_DATE}.json'
print('prediction file exists:', os.path.exists(pred_path), pred_path)

if os.path.exists(pred_path):
    with open(pred_path, 'r', encoding='utf-8') as f:
        pred = json.load(f)
    print('prediction_date:', pred.get('prediction_date'))
    print('summary:', pred.get('execution_summary'))

print('Drive artifact roots:')
for d in ['optuna_models', 'optuna_results', 'optuna_logs', 'outputs']:
    p = f'{DRIVE_ROOT}/{d}'
    print('-', p, 'exists=' + str(os.path.exists(p)))
